# Evaluate SpAM Simulation (task-v3)

Read-only evaluation of a **task-v3** run (the generative, coordinate-space model: per-subject PC "perspective" + a local 2-D arrangement projection per trial). This notebook is v3-only - older v0.1/v2.3/v2.4 runs keep their own `evaluation*.ipynb` notebooks.

Levers: `num_subjects`, `trials_per_subject`, `images_per_trial`, `subjects_noise_scale`, `subjects_noise_df`, `frac_trials_repeated` (whole-trial test-retest repeats), and `perspective_dispersion` (between-subject disagreement). The ground-truth spectrum (`use_isotropic`/`decay`/`n_clusters`) is fixed per run, not swept. Figures below read only the four small result CSVs a completed run wrote.

In [ ]:
import plotly.io as pio

from SpAM_Simulations import eval_helpers as eh

pio.renderers.default = "browser"

## Load Run
Set `RUN_RESULTS_DIR` to a folder name under `SpAM_Simulations/sim_results/` (e.g. `"task-v2.3"`).
`eh.load_run` resolves it relative to `eval_helpers.py`'s own directory (not the notebook
kernel's cwd) and raises `FileNotFoundError` naming every missing file if the run directory or
any of the four expected files is absent.

In [ ]:
# Bundled demo run (small; MDS via a numpy classical-MDS stand-in, not R/SMACOF).
# Point this at your own completed run (e.g. "sim_results/task-v3-isotropic") to analyse it.
RUN_RESULTS_DIR = "sim_results/task-v3-smoke"
run = eh.load_run(RUN_RESULTS_DIR)
print(f"task_version={run.task_version}, {len(run.mds_meta)} MDS rows, {len(run.coverage)} coverage rows")

### Levers in this run

In [ ]:
eh.lever_summary_table(run.levers).show()

## Coverage
Two coverage metrics, read directly from `out/coverage.csv` (no recomputation):
- **% Images Seen** (`img_coverage`): percentage of images observed at least once.
- **% Pairs Seen** (`pair_coverage`): percentage of image-pairs observed at least once.

Layout: rows = the two metrics above; columns = `trials_per_subject`; x-axis = `num_subjects`;
traces = `images_per_trial` and `frac_images_repeated` (the latter dropped automatically on a
task-v0.1 run, where it doesn't exist).

`metrics.coverage()` computes both quantities purely from `num_obs` (which images/pairs were
ever *drawn*), never from the noisy `distances` values - so both metrics are mathematically
independent of `subjects_noise_scale`/`subjects_noise_df`. They still vary stochastically
across `rep`, since which images land in which trial is itself a random draw - hence we
aggregate with mean +/- SEM across `rep` (and across the noise levers, which contribute no
additional variance here) before plotting.

In [ ]:
GROUP_COLS = [c for c in ["num_subjects", "trials_per_subject", "images_per_trial", "perspective_dispersion", "frac_trials_repeated"]
              if c in run.coverage.columns]
coverage_summary = (
    run.coverage.groupby(GROUP_COLS)
    .agg(img_coverage_mean=("img_coverage", "mean"), img_coverage_sem=("img_coverage", "sem"),
         pair_coverage_mean=("pair_coverage", "mean"), pair_coverage_sem=("pair_coverage", "sem"))
    .reset_index()
)
eh.faceted_metric_figure(
    coverage_summary, x="num_subjects",
    metrics=[("img_coverage_mean", "img_coverage_sem", "% Images Seen"),
             ("pair_coverage_mean", "pair_coverage_sem", "% Pairs Seen")],
    col_by="trials_per_subject", trace_by=["images_per_trial", "perspective_dispersion", "frac_trials_repeated"],
    title="Coverage by Experimental Configuration", x_title="Number of Subjects",
).show()

## Connectivity (P[Connected])
`num_connected_components == 1` checks whether the observed image-pair graph is a single
connected component - a strict prerequisite for MDS. Same independence property as coverage
above: connectivity depends only on which pairs were drawn (`num_obs`), not on the noisy
distance values, so it too is independent of `subjects_noise_scale`/`subjects_noise_df` (but
still varies across `rep`, since the draws themselves are random).

Layout: y = % of reps with exactly 1 connected component; x = `num_subjects`; columns =
`trials_per_subject`; rows = `images_per_trial`; traces = `frac_images_repeated` (dropped on a
task-v0.1 run).

In [ ]:
conn = run.coverage.copy()
conn["is_connected"] = conn["num_connected_components"] == 1
GROUP_COLS = [c for c in ["num_subjects", "trials_per_subject", "images_per_trial", "perspective_dispersion", "frac_trials_repeated"]
              if c in conn.columns]
connectivity_summary = (
    conn.groupby(GROUP_COLS)
    .agg(p_connected_mean=("is_connected", "mean"), p_connected_sem=("is_connected", "sem"))
    .reset_index()
)
eh.faceted_lever_figure(
    connectivity_summary, x="num_subjects", y="p_connected_mean", y_sem="p_connected_sem",
    row_by="images_per_trial", col_by="trials_per_subject", trace_by=["perspective_dispersion", "frac_trials_repeated"],
    title="P[Connected] by Experimental Configuration", x_title="Number of Subjects",
    y_title="% of reps connected",
).show()

## Test-Retest Reliability (task-v2.4)
For task-v2.4 runs (which sweep `frac_trials_repeated`), each subject who received whole-trial
repeats yields a test-retest reliability: the mean Spearman correlation between the original and
repeat presentations of their repeated trials (`mean_test_retest` in `out/coverage.csv`). The
figure shows how it tracks `subjects_noise_scale`, one trace per `frac_trials_repeated` (the
all-NaN `frac_trials_repeated = 0` slice is omitted). Skipped automatically on task-v0.1/v2.3
runs, which don't carry this metric.


In [ ]:
if "mean_test_retest" in run.coverage.columns:
    eh.test_retest_figure(run.coverage).show()
else:
    print("No test-retest data in this run (needs frac_trials_repeated > 0).")


## Pre-MDS Stability
Spearman rank correlation between the mean observed distances of different repetitions of the
same configuration (`out/stability.csv`) - the data-reliability ceiling before MDS is even run.

Layout: y = mean Spearman correlation; x = `num_subjects`; columns = `trials_per_subject`;
rows = `images_per_trial`; traces = `subjects_noise_scale`, `subjects_noise_df`,
`perspective_dispersion`, `frac_trials_repeated` (any constant or missing feature is dropped
from the trace name and captioned instead).

In [ ]:
GROUP_COLS = [c for c in eh.LEVER_COLUMNS if c in run.stability.columns]
stability_summary = (
    run.stability.dropna(subset=["spearman"])
    .groupby(GROUP_COLS)
    .agg(spearman_mean=("spearman", "mean"), spearman_sem=("spearman", "sem"))
    .reset_index()
)
eh.faceted_lever_figure(
    stability_summary, x="num_subjects", y="spearman_mean", y_sem="spearman_sem",
    row_by="images_per_trial", col_by="trials_per_subject",
    trace_by=["subjects_noise_scale", "subjects_noise_df", "perspective_dispersion", "frac_trials_repeated"],
    title="Pre-MDS Stability by Experimental Configuration", x_title="Number of Subjects",
    y_title="Spearman R",
).show()

## MDS Scree Plots
MDS stress vs. target dimensionality, read directly from `mds_store/meta.csv` (successful
runs only - `status` in `{success, max_iters}`).

Layout: y = mean stress; x = `ndim`; columns = `trials_per_subject`; rows = `images_per_trial`;
traces = `num_subjects`, `subjects_noise_scale`, `subjects_noise_df` (a trace-feature with only
one value in this run is removed from the trace name and reported in the figure caption
instead). One separate figure is produced per varying **condition** lever combination
(`frac_trials_repeated` and/or `perspective_dispersion`), labelled in its title, rather than
pooling across them.

In [ ]:
success = run.mds_meta[run.mds_meta["status"].isin(["success", "max_iters"])]
GROUP_COLS = [c for c in ["num_subjects", "trials_per_subject", "images_per_trial",
                          "subjects_noise_scale", "subjects_noise_df", "ndim"]
              if c in success.columns]

# Slice by whichever condition lever(s) vary - frac_trials_repeated and/or
# perspective_dispersion - so panels never pool (or duplicate markers) across them.
for caption, df in eh.condition_slices(success):
    stress_summary = df.groupby(GROUP_COLS)["stress"].agg(stress_mean="mean", stress_sem="sem").reset_index()
    title = "MDS Stress by Dimension" + (f" ({caption})" if caption else "")
    eh.faceted_lever_figure(
        stress_summary, x="ndim", y="stress_mean", y_sem="stress_sem",
        row_by="images_per_trial", col_by="trials_per_subject",
        trace_by=["num_subjects", "subjects_noise_scale", "subjects_noise_df"],
        title=title, x_title="Target Dimensionality", y_title="Stress",
    ).show()

## Post-MDS (Embedding) Stability
Mean Spearman agreement of reconstructed MDS distances (`confdist`) across repetitions,
already computed by `pipeline.compute_embedding_stability` and stored in
`out/embedding_stability.csv` - no recomputation, no touching `confdists.f32`.

Same layout and condition-lever-slicing rules as the scree plots above: y = `mean_spearman`;
x = `ndim`; columns = `trials_per_subject`; rows = `images_per_trial`; traces = `num_subjects`,
`subjects_noise_scale`, `subjects_noise_df`; one figure per `frac_trials_repeated` /
`perspective_dispersion` combination.

In [ ]:
# Same per-condition-lever slicing as the scree plots above (frac_trials_repeated and/or
# perspective_dispersion), so each (num_subjects, ndim) cell holds exactly one point.
for caption, df in eh.condition_slices(run.embedding_stability):
    title = "Embedding Stability by Dimension" + (f" ({caption})" if caption else "")
    eh.faceted_lever_figure(
        df, x="ndim", y="mean_spearman", y_sem="sem_spearman",
        row_by="images_per_trial", col_by="trials_per_subject",
        trace_by=["num_subjects", "subjects_noise_scale", "subjects_noise_df"],
        title=title, x_title="Target Dimensionality", y_title="Spearman R",
    ).show()

## Drill-Down: Focus Configuration
The overview figures above average over many configurations at once. Here we fix every lever
except `num_subjects` to a single value (the focus configuration) and re-examine coverage, MDS
convergence, and stability for just that slice - mirroring `evaluation.ipynb`'s "Final
Configuration Evaluation" section, but reading from the same pre-loaded `run` instead of
recomputing anything.

Available combinations of every lever except `num_subjects` (from `mds_store/meta.csv`):

In [ ]:
SECONDARY_LEVERS = [l for l in eh.LEVER_COLUMNS if l != "num_subjects" and l in run.mds_meta.columns]
eh.available_configs(run.mds_meta, SECONDARY_LEVERS)

In [ ]:
# Pick one configuration from the available-configs table above and EDIT the values
# to match a row. Any key not present in this run is dropped automatically. Pinning
# perspective_dispersion and frac_trials_repeated keeps the drill-down from pooling
# across them. Defaults below match the run_task_v3_sim.sh grid.
FOCUS_CONFIG = {
    "trials_per_subject": 10, "images_per_trial": 20,
    "subjects_noise_scale": 0.5, "subjects_noise_df": 1,
    "perspective_dispersion": 0.0, "frac_trials_repeated": 0.1,
}
FOCUS_CONFIG = {k: v for k, v in FOCUS_CONFIG.items() if k in run.mds_meta.columns}
assert not eh.filter_to_config(run.mds_meta, FOCUS_CONFIG).empty, (
    f"No rows match {FOCUS_CONFIG} - check the available-configs table above"
)
FOCUS_CONFIG

#### Coverage

In [ ]:
cov = eh.filter_to_config(run.coverage, FOCUS_CONFIG)
cov_summary = (
    cov.groupby("num_subjects")
    .agg(img_coverage_mean=("img_coverage", "mean"), img_coverage_sem=("img_coverage", "sem"),
         pair_coverage_mean=("pair_coverage", "mean"), pair_coverage_sem=("pair_coverage", "sem"))
    .reset_index()
)
eh.faceted_metric_figure(
    cov_summary, x="num_subjects",
    metrics=[("img_coverage_mean", "img_coverage_sem", "% Images Seen"),
             ("pair_coverage_mean", "pair_coverage_sem", "% Pairs Seen")],
    title="Coverage - Focus Configuration", x_title="Number of Subjects",
).show()

#### Convergence

In [ ]:
meta = eh.filter_to_config(run.mds_meta, FOCUS_CONFIG)
eh.convergence_bar_figure(meta).show()

#### Stability: Pre- vs. Post-MDS

In [ ]:
emb = eh.filter_to_config(run.embedding_stability, FOCUS_CONFIG)
stab = eh.filter_to_config(run.stability, FOCUS_CONFIG)
eh.pre_post_mds_stability_figure(emb, stab).show()

## Required Subjects (plateau-N)
With per-subject `perspective_dispersion` the stability-vs-N curve saturates *below* 1.0, so the convergence target is each curve's own plateau, not a fixed Spearman threshold. `eh.plateau_num_subjects` reports, per target `ndim`, the smallest `num_subjects` within `tol` of the asymptote (read at the largest swept N). If `plateau_num_subjects == max_num_subjects` the sweep has **not** saturated and needs a larger N.

In [ ]:
# One row per (ndim [, repetition/perspective lever]) with the smallest N reaching the plateau.
GROUP = [c for c in ["ndim", "frac_trials_repeated", "perspective_dispersion"]
         if c in run.embedding_stability.columns]
plateau = eh.plateau_num_subjects(run.embedding_stability, group_by=GROUP, tol=0.01)
plateau.sort_values(GROUP)